# EP1: 제조 데이터로 지식그래프 만들기

**유튜브 시리즈 '제조 GraphRAG 실전' -- Episode 1**

---

## 학습 목표

1. **온톨로지 설계 이해** -- 엔티티 타입 5종, 관계 타입 7종의 구조를 파악합니다
2. **Neo4j에 7노드 그래프 구축** -- Cypher로 제조 Knowledge Graph를 직접 만듭니다
3. **3-hop 근본원인 분석 체험** -- "접착 박리 결함의 원인 설비는?"에 답합니다

## 소요 시간

| 구분 | 시간 |
|------|------|
| 영상 시청 | ~20분 |
| 실습 | ~30분 |
| **합계** | **~50분** |

---

## 목차

1. 환경 설정 (Setup)
2. 왜 그래프인가? (Why Graphs?)
3. 온톨로지 설계 (Ontology Design)
4. Neo4j에 그래프 만들기 (Build the Graph)
5. Cypher 쿼리로 인사이트 발굴
6. Vector RAG vs GraphRAG 비교
7. 다음 에피소드 미리보기
8. 세션 정리

---
## 1. 환경 설정

### 1.1 필수 패키지 설치 확인

Neo4j Python 드라이버와 환경 변수 관리를 위한 `python-dotenv`가 필요합니다.  
아래 셀을 실행하여 설치 여부를 확인하세요.

In [ ]:
# 필수 패키지 설치 확인 및 설치
import subprocess
import sys

def check_and_install(package_name, import_name=None):
    """패키지가 설치되어 있는지 확인하고, 없으면 설치합니다."""
    import_name = import_name or package_name
    try:
        __import__(import_name)
        print(f"[OK] {package_name} 설치 확인 완료")
    except ImportError:
        print(f"[설치 중] {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"[OK] {package_name} 설치 완료")

check_and_install("neo4j")
check_and_install("python-dotenv", "dotenv")

### 1.2 환경 변수 로드

프로젝트 루트에 `.env` 파일을 만들고 Neo4j 접속 정보를 넣어주세요.

```
NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=your_password_here
```

> **참고**: Neo4j Desktop 또는 AuraDB Free Tier를 사용할 수 있습니다.  
> AuraDB의 경우 URI가 `neo4j+s://xxxxx.databases.neo4j.io` 형식입니다.

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일 로드 (상위 디렉토리에서도 탐색)
load_dotenv()
load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j User: {NEO4J_USER}")
print(f"Password 설정: {'OK' if NEO4J_PASSWORD else 'NOT SET - .env 파일을 확인하세요'}")

### 1.3 Neo4j 연결 테스트

드라이버를 생성하고 연결이 정상인지 확인합니다.

In [ ]:
from neo4j import GraphDatabase

# Neo4j 드라이버 생성
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# 연결 테스트
try:
    driver.verify_connectivity()
    print("[OK] Neo4j 연결 성공!")

    # 서버 정보 확인
    with driver.session() as session:
        result = session.run("RETURN 'Hello, Neo4j!' AS message")
        record = result.single()
        print(f"서버 응답: {record['message']}")
except Exception as e:
    print(f"[ERROR] 연결 실패: {e}")
    print("Neo4j가 실행 중인지, .env 설정이 올바른지 확인하세요.")

In [ ]:
# 편의 함수: Cypher 쿼리 실행 및 결과 출력
def run_query(query, parameters=None, print_result=True):
    """Cypher 쿼리를 실행하고 결과를 반환합니다."""
    with driver.session() as session:
        result = session.run(query, parameters or {})
        records = [record.data() for record in result]
        if print_result:
            if records:
                for i, rec in enumerate(records, 1):
                    print(f"  [{i}] {rec}")
            else:
                print("  (결과 없음)")
        return records

print("[OK] run_query 함수 준비 완료")

---
## 2. 왜 그래프인가?

### ChatGPT에게 물어봤습니다

다음 질문을 생각해보세요:

> **"접착 박리 결함의 원인 설비는 무엇이고, 그 설비를 사용하는 공정의 원자재는?"**

이 질문에 답하려면 **3단계(3-hop)**의 추론이 필요합니다:

```
접착 박리 → (원인 공정) → 열압착 → (사용 설비) → HP-01
열압착 → (이전 공정) → 혼합 → (투입 원자재) → 마찰재
```

### 문제: 정보가 여러 문서에 흩어져 있습니다

| 정보 | 출처 |
|------|------|
| 접착 박리 → 열압착 공정 원인 | 품질보고서 |
| 열압착 → HP-01 설비 사용 | 공정매뉴얼 |
| 혼합 공정 → 마찰재 투입 | 자재관리 |

벡터 검색(Vector RAG)은 "접착 박리 설비 원자재"로 검색해도  
3개의 서로 다른 문서를 **올바른 순서로** 연결하지 못합니다.

**Knowledge Graph가 이 문제를 해결합니다.**  
노드와 관계로 구조화하면, 한 줄의 Cypher 쿼리로 정확한 답을 얻을 수 있습니다.

---
## 3. 온톨로지 설계

### 온톨로지(Ontology)란?

온톨로지는 그래프의 **설계도**입니다.  
어떤 종류의 노드(엔티티)가 있고, 어떤 관계로 연결되는지를 정의합니다.

- **엔티티 타입**: 노드의 종류 (예: 공정, 설비, 결함)
- **관계 타입**: 노드 간 연결 방식 (예: NEXT, USES_EQUIPMENT)

### Stage 0 미니 데모: 7노드 그래프 구조

```
[혼합] --NEXT--> [열압착] --NEXT--> [연마]
  |                 |
  |USES_MATERIAL    |USES_EQUIPMENT
  v                 v
[마찰재]         [HP-01]
                    ^
                    |CAUSED_BY_EQUIPMENT
[접착 박리] --------+
  |                 |CAUSED_BY_PROCESS
  |DETECTED_AT      +-------> [열압착]
  v
[전단강도 시험] --INSPECTS--> [열압착]
```

### 5가지 엔티티 타입

| 레이블 | 한글 | ID | 이름 | 설명 |
|--------|------|-----|------|------|
| Component | 원자재 | CMP-001 | 마찰재 (Friction Material) | 브레이크패드 핵심 원자재 |
| Process | 공정 | PRC-001, PRC-003, PRC-005 | 혼합, 열압착, 연마 | 제조 공정 3단계 |
| Equipment | 설비 | EQP-003 | 열압착 프레스 HP-01 | 열압착 공정 전용 설비 |
| Defect | 결함 | DEF-001 | 접착 박리 (Delamination) | 심각도: Critical |
| Inspection | 검사 | INS-001 | 전단강도 시험 (Shear Strength Test) | 접착 품질 검증 |

### 7가지 관계 타입

| 관계 | 한글 | 출발 → 도착 | 설명 |
|------|------|-------------|------|
| NEXT | 다음 공정 | Process → Process | 공정 흐름 (혼합→열압착→연마) |
| USES_EQUIPMENT | 설비 사용 | Process → Equipment | 열압착→HP-01 |
| USES_MATERIAL | 원자재 투입 | Process → Component | 혼합→마찰재 |
| DETECTED_AT | ~에서 발견됨 | Defect → Inspection | 접착 박리→전단강도 시험 |
| INSPECTS | ~를 검사함 | Inspection → Process | 전단강도 시험→열압착 |
| CAUSED_BY_PROCESS | 원인 공정 | Defect → Process | 접착 박리→열압착 |
| CAUSED_BY_EQUIPMENT | 원인 설비 | Defect → Equipment | 접착 박리→HP-01 |

---
## 4. Neo4j에 그래프 만들기

온톨로지 설계를 바탕으로 실제 Neo4j에 7개 노드와 8개 관계를 생성합니다.

### 4a. 기존 데이터 초기화

In [ ]:
run_query("MATCH (n) DETACH DELETE n", print_result=False)
print("[OK] 기존 데이터 초기화 완료")

### 4b. 7개 노드 생성

원자재 1개 + 공정 3개 + 설비 1개 + 결함 1개 + 검사 1개 = **7개 노드**

In [ ]:
# === 7개 노드 생성 ===
print("=== 7개 노드 생성 ===")

run_query("""
    // 원자재 (Component)
    CREATE (cmp001:Component {
        component_id: 'CMP-001',
        name: '마찰재 (Friction Material)'
    })

    // 공정 (Process) - 3개
    CREATE (prc001:Process {
        process_id: 'PRC-001',
        name: '혼합 (Mixing)'
    })
    CREATE (prc003:Process {
        process_id: 'PRC-003',
        name: '열압착 (Hot Press)'
    })
    CREATE (prc005:Process {
        process_id: 'PRC-005',
        name: '연마 (Grinding)'
    })

    // 설비 (Equipment)
    CREATE (eqp003:Equipment {
        equipment_id: 'EQP-003',
        name: '열압착 프레스 HP-01 (Hot Press HP-01)'
    })

    // 결함 (Defect)
    CREATE (def001:Defect {
        defect_id: 'DEF-001',
        name: '접착 박리 (Delamination)',
        severity: 'Critical'
    })

    // 검사 (Inspection)
    CREATE (ins001:Inspection {
        inspection_id: 'INS-001',
        name: '전단강도 시험 (Shear Strength Test)'
    })

    RETURN
        cmp001.name AS 원자재,
        prc001.name AS 공정1_혼합,
        prc003.name AS 공정2_열압착,
        prc005.name AS 공정3_연마,
        eqp003.name AS 설비,
        def001.name AS 결함,
        ins001.name AS 검사
""")

print("\n7개 노드 생성 완료!")

### 4c. 8개 관계 생성

| # | 관계 | 의미 |
|---|------|------|
| 1 | 혼합 -[NEXT]-> 열압착 | 공정 흐름 |
| 2 | 열압착 -[NEXT]-> 연마 | 공정 흐름 |
| 3 | 열압착 -[USES_EQUIPMENT]-> HP-01 | 설비 사용 |
| 4 | 혼합 -[USES_MATERIAL]-> 마찰재 | 원자재 투입 |
| 5 | 접착 박리 -[DETECTED_AT]-> 전단강도 시험 | 결함 발견 |
| 6 | 전단강도 시험 -[INSPECTS]-> 열압착 | 공정 검사 |
| 7 | 접착 박리 -[CAUSED_BY_PROCESS]-> 열압착 | 결함 원인 (공정) |
| 8 | 접착 박리 -[CAUSED_BY_EQUIPMENT]-> HP-01 | 결함 원인 (설비) |

In [ ]:
# === 8개 관계 생성 ===
print("=== 8개 관계 생성 ===")

run_query("""
    // 기존 노드 매칭
    MATCH (cmp001:Component   {component_id: 'CMP-001'})
    MATCH (prc001:Process     {process_id: 'PRC-001'})
    MATCH (prc003:Process     {process_id: 'PRC-003'})
    MATCH (prc005:Process     {process_id: 'PRC-005'})
    MATCH (eqp003:Equipment   {equipment_id: 'EQP-003'})
    MATCH (def001:Defect      {defect_id: 'DEF-001'})
    MATCH (ins001:Inspection  {inspection_id: 'INS-001'})

    // 1-2. 공정 흐름: 혼합 -> 열압착 -> 연마
    CREATE (prc001)-[:NEXT]->(prc003)
    CREATE (prc003)-[:NEXT]->(prc005)

    // 3. 열압착 공정 -> HP-01 설비 사용
    CREATE (prc003)-[:USES_EQUIPMENT]->(eqp003)

    // 4. 혼합 공정 -> 마찰재 투입
    CREATE (prc001)-[:USES_MATERIAL]->(cmp001)

    // 5. 접착 박리 -> 전단강도 시험에서 발견
    CREATE (def001)-[:DETECTED_AT]->(ins001)

    // 6. 전단강도 시험 -> 열압착 공정 검증
    CREATE (ins001)-[:INSPECTS]->(prc003)

    // 7. 접착 박리 <- 열압착 공정이 원인
    CREATE (def001)-[:CAUSED_BY_PROCESS]->(prc003)

    // 8. 접착 박리 <- HP-01 설비가 원인
    CREATE (def001)-[:CAUSED_BY_EQUIPMENT]->(eqp003)

    RETURN '관계 8개 생성 완료' AS status
""")

print("\n관계 생성 요약:")
print("  [1] 혼합 -[NEXT]-> 열압착           : 공정 흐름")
print("  [2] 열압착 -[NEXT]-> 연마            : 공정 흐름")
print("  [3] 열압착 -[USES_EQUIPMENT]-> HP-01 : 설비 사용")
print("  [4] 혼합 -[USES_MATERIAL]-> 마찰재   : 원자재 투입")
print("  [5] 접착 박리 -[DETECTED_AT]-> 전단강도 시험 : 결함 발견")
print("  [6] 전단강도 시험 -[INSPECTS]-> 열압착       : 공정 검사")
print("  [7] 접착 박리 -[CAUSED_BY_PROCESS]-> 열압착  : 결함 원인(공정)")
print("  [8] 접착 박리 -[CAUSED_BY_EQUIPMENT]-> HP-01 : 결함 원인(설비)")

### 4d. 그래프 검증

In [ ]:
# === 그래프 검증 ===
print("=== 그래프 검증 ===")

print("\n[1] 전체 노드 수:")
run_query("MATCH (n) RETURN count(n) AS 전체_노드_수")

print("\n[2] 전체 관계 수:")
run_query("MATCH ()-[r]->() RETURN count(r) AS 전체_관계_수")

print("\n[3] 레이블별 노드 수:")
run_query("""
    MATCH (n)
    RETURN labels(n)[0] AS 레이블, count(n) AS 노드수
    ORDER BY 노드수 DESC
""")

print("\n[4] 전체 관계 목록:")
run_query("""
    MATCH (a)-[r]->(b)
    RETURN labels(a)[0] + ':' + a.name AS 출발,
           type(r) AS 관계,
           labels(b)[0] + ':' + b.name AS 도착
""")

print("\n** Neo4j Browser에서 'MATCH (n)-[r]->(m) RETURN n,r,m' 실행하면 시각적으로 볼 수 있습니다! **")

---
## 5. Cypher 쿼리로 인사이트 발굴

7개 노드, 8개 관계만으로도 의미 있는 분석이 가능합니다.  
1-hop부터 3-hop까지 단계별로 쿼리해봅니다.

### 질의 1: 전체 그래프 확인

In [ ]:
# 질의 1: 전체 그래프 확인
print("=" * 60)
print("질의 1: 전체 그래프 확인")
print("=" * 60)

print("\n[노드 목록]")
run_query("""
    MATCH (n)
    RETURN labels(n)[0] AS 레이블, n.name AS 이름
    ORDER BY labels(n)[0]
""")

print("\n[관계 목록]")
run_query("""
    MATCH (a)-[r]->(b)
    RETURN a.name AS 출발, type(r) AS 관계, b.name AS 도착
""")

### 질의 2: 공정 흐름 확인

NEXT 관계를 따라 전체 공정 순서를 확인합니다.

In [ ]:
# 질의 2: 공정 흐름
print("=" * 60)
print("질의 2: 공정 흐름 (혼합 -> 열압착 -> 연마)")
print("=" * 60)

run_query("""
    MATCH path = (start:Process)-[:NEXT*]->(end:Process)
    WHERE NOT ()<-[:NEXT]-(start)
    RETURN [n IN nodes(path) | n.name] AS 공정흐름
""")

print("\n--> NEXT 관계를 따라가면 전체 공정 순서를 파악할 수 있습니다.")

### 질의 3: 1-hop 질문 (벡터 RAG로 충분)

> **"열압착 공정에서 사용하는 설비는?"**

1-hop 질문은 단일 문서에 답이 있으므로 벡터 검색으로도 답할 수 있습니다.

In [ ]:
# 질의 3: 1-hop - 열압착 공정의 설비
print("=" * 60)
print("질의 3 (1-hop): 열압착 공정에서 사용하는 설비는?")
print("=" * 60)

run_query("""
    MATCH (p:Process {name: '열압착 (Hot Press)'})-[:USES_EQUIPMENT]->(e:Equipment)
    RETURN p.name AS 공정, e.name AS 설비
""")

print("\n--> 이 정도는 벡터 검색으로도 답할 수 있습니다.")
print("    '열압착 설비'로 검색하면 관련 문서 청크를 찾을 수 있으니까요.")

### 질의 4: 3-hop 근본원인 추적

# ========================================
# 핵심 쿼리: 3-hop 근본원인 추적
# ========================================

> **"접착 박리 결함의 원인 공정과 설비는?"**

경로: `결함 → 검사 → 공정 → 설비`

```
접착 박리 --DETECTED_AT--> 전단강도 시험 --INSPECTS--> 열압착 --USES_EQUIPMENT--> HP-01
```

이것이 GraphRAG의 핵심 가치입니다!

In [ ]:
# =============================================
# 핵심 쿼리: 3-hop 근본원인 추적
# =============================================
print("=" * 60)
print("질의 4 (3-hop): 접착 박리 결함의 원인 공정과 설비는?")
print("=" * 60)

run_query("""
    MATCH (d:Defect {name: '접착 박리 (Delamination)'})-[:DETECTED_AT]->(i:Inspection)
          -[:INSPECTS]->(p:Process)-[:USES_EQUIPMENT]->(e:Equipment)
    RETURN d.name AS 결함명,
           d.severity AS 심각도,
           i.name AS 발견검사,
           p.name AS 원인공정,
           e.name AS 원인설비
""")

print("\n" + "*" * 60)
print("이것이 GraphRAG의 핵심 가치!")
print("벡터 RAG로는 이 3-hop 관계를 추적할 수 없습니다.")
print("*" * 60)
print("\n경로 해석:")
print("  접착 박리 (결함)")
print("    --[DETECTED_AT]--> 전단강도 시험 (검사)")
print("    --[INSPECTS]--> 열압착 (공정)")
print("    --[USES_EQUIPMENT]--> HP-01 (설비)")

### 질의 5: 설비 영향 분석

> **"HP-01 설비가 관련된 모든 것을 보여줘"**

In [ ]:
# 질의 5: 설비 영향 분석
print("=" * 60)
print("질의 5: HP-01 설비 영향 분석")
print("=" * 60)

run_query("""
    MATCH (e:Equipment {name: '열압착 프레스 HP-01 (Hot Press HP-01)'})<-[r]-(connected)
    RETURN e.name AS 설비,
           type(r) AS 관계,
           labels(connected)[0] AS 노드유형,
           connected.name AS 연결노드
""")

print("\n--> 하나의 설비에 공정, 결함이 모두 연결되어 있습니다.")
print("    설비 교체/점검 시 영향 범위를 즉시 파악할 수 있습니다.")

### 질의 6: 전체 경로 시각화

> **"접착 박리 결함을 중심으로 전체 연결 그래프를 보여줘"**

Neo4j Browser에서 이 쿼리를 실행하면 결함을 중심으로 뻗어나가는 전체 그래프를 시각화할 수 있습니다.

In [ ]:
# 질의 6: 전체 경로 시각화
print("=" * 60)
print("질의 6: 접착 박리 중심 전체 경로 탐색 (1~4 hop)")
print("=" * 60)

run_query("""
    MATCH path = (d:Defect)-[*1..4]-(connected)
    WHERE d.name = '접착 박리 (Delamination)'
    RETURN DISTINCT
        connected.name AS 연결노드,
        labels(connected)[0] AS 노드유형,
        length(path) AS hop수
    ORDER BY hop수
""")

print("\n** Neo4j Browser에서 아래 쿼리를 실행해보세요: **")
print("   MATCH path = (d:Defect)-[*1..4]-(connected)")
print("   WHERE d.name = '접착 박리 (Delamination)'")
print("   RETURN path")

---
## 6. Vector RAG vs GraphRAG 비교

이번 실습에서 체험한 내용을 정리합니다.

### 질문 유형별 비교

| 구분 | Vector RAG | GraphRAG |
|------|-----------|----------|
| **1-hop 질문** (열압착 설비는?) | O 정확 | O 정확 |
| **2-hop 질문** (결함 원인 공정은?) | ? 불확실 | O 정확 |
| **3-hop 질문** (결함→검사→공정→설비) | X 불가능 | O 정확 |
| **근본원인 추적** | X | O |
| **경로 탐색** | X | O |
| **영향 분석** | X | O |

### 왜 Vector RAG는 Multi-hop에 실패하는가?

| 단계 | 필요한 정보 | 문서 위치 |
|------|------------|----------|
| 1단계 | 접착 박리 → 전단강도 시험에서 발견 | 품질보고서_chunk_47 |
| 2단계 | 전단강도 시험 → 열압착 공정 검증 | 검사매뉴얼_chunk_23 |
| 3단계 | 열압착 → HP-01 설비 사용 | 공정매뉴얼_chunk_12 |

**Vector RAG의 한계:**
- 3개의 서로 다른 문서, 서로 다른 청크에 정보가 흩어져 있음
- "접착 박리 원인 설비"로 검색해도 3개 청크를 올바른 순서로 연결 불가
- 설령 모든 청크가 검색되더라도, LLM이 정확한 체인을 추론해야 함

**GraphRAG의 해결:**
```cypher
MATCH (d:Defect)-[:DETECTED_AT]->(i)-[:INSPECTS]->(p)-[:USES_EQUIPMENT]->(e)
RETURN d, i, p, e
```
한 줄의 Cypher로 **구조적으로 보장된** 정확한 답을 얻습니다.

---
## 7. 다음 에피소드 미리보기

### EP2: Text2Cypher로 자연어 질의
- LLM이 자연어를 Cypher 쿼리로 변환하는 GraphRAG 파이프라인을 구축합니다
- "접착 박리 원인이 뭐야?" → 자동으로 3-hop Cypher 생성 → 정확한 답변

### EP3: Agentic GraphRAG 완성 + 실전 팁
- Agent가 질문을 분석하고 최적의 쿼리 전략을 선택합니다
- 실제 제조 현장에서 쓸 수 있는 실전 팁을 공유합니다

### 확장 로드맵: 7개 노드에서 5,000+개로

| 단계 | 노드 수 | 설명 |
|------|---------|------|
| Stage 0 (이번 실습) | 7 | 미니 데모 - 온톨로지 체험 |
| Stage 1 | 35 | 교육용 - 결함 4종, 공정 7단계, 설비 7대 |
| Stage 2 | 1,000 | 프로토타입 - 실제 데이터 시뮬레이션 |
| Stage 3 | 5,000+ | 프로덕션 - 실시간 모니터링 가능 수준 |

> **"7개 노드 -> 35개 -> 1,000개로 확장하면서 진짜 파이프라인을 만들어봅니다"**

---
## 8. 세션 정리

In [ ]:
# 세션 정리: 드라이버 종료
# 실습이 모두 끝난 후 실행하세요.
# (다음 실습을 이어서 할 경우에는 실행하지 마세요)

# driver.close()
# print("[OK] Neo4j 드라이버 종료 완료")

print("=" * 60)
print("EP1 실습을 마칩니다.")
print("=" * 60)
print("\n오늘 배운 것:")
print("  [1] 온톨로지 설계 - 엔티티 5종 + 관계 7종")
print("  [2] Neo4j에 7노드 그래프 구축")
print("  [3] 3-hop 근본원인 분석의 위력")
print("  [4] Vector RAG vs GraphRAG 차이")
print("\n다음 에피소드에서 만나요!")
print("EP2: Text2Cypher로 자연어 질의 (GraphRAG 파이프라인 구축)")